# Construindo um Assistente com RAG, Ollama e Streamlit
**Webinário CIIA — Encontro 2** · Roger Quinelato (condução) · João Victor Rikio Enomoto (suporte)

| Bloco | Conteúdo | Min |
|---|---|---|
| 1 | Recapitulando o pipeline RAG + ambiente | 10 |
| 2 | Indexação: PDFs → chunks → embeddings → ChromaDB (+ metadados com LLM) | 18 |
| 3 | Retrieval top-k na prática: k, distâncias, filtros, cross-lingual | 15 |
| 3b | Busca em dois estágios: resumos → chunks | 7 |
| 4 | SHAP: explicando o retrieval | 12 |
| 5 | Prompt augmentation: respostas com e sem contexto | 12 |
| 6 | Integração com LLM local usando Ollama | 12 |
| 7 | Construção do chatbot com Streamlit | 15 |
| 8 | Avaliação (RAGAS), reranking e próximos passos | 8 |

> Antes de rodar: siga o `README.md` (Ollama, modelos, `.venv`). Toda etapa lenta tem uma **saída pré-computada**
> em `resultados/`; as chaves abaixo decidem se a célula roda ao vivo ou só carrega o resultado salvo.

In [1]:
import json
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))
import config
import rag

REINDEXAR = False        # True: apaga e recria a coleção (alguns minutos em CPU)
LLM_AO_VIVO = True       # False: usa as respostas salvas em resultados/
SHAP_AO_VIVO = True      # False: mostra o gráfico SHAP salvo em resultados/

warnings.filterwarnings("ignore", message="IProgress not found")
pd.set_option("display.max_colwidth", 120)


def tabela(resultados, colunas=("posicao", "distancia", "arquivo", "pagina", "ano", "tema", "idioma", "texto")):
    if not resultados:
        return pd.DataFrame(columns=list(colunas))
    df = pd.DataFrame(resultados)[list(colunas)]
    if "texto" in df:
        df["texto"] = df["texto"].str.slice(0, 110) + "…"
    return df.round({"distancia": 4})


def carregar_resultado(nome):
    return json.loads((config.PASTA_RESULTADOS / nome).read_text(encoding="utf-8"))

## Bloco 1 — Recapitulando o pipeline RAG

```
PDFs ──► texto por página ──► chunks ──► embeddings (bge-m3) ──► ChromaDB
                                                                     │
pergunta ──► embedding ──► top-k (+ filtros de metadados) ◄──────────┘
                              │
                              ▼
               prompt = instruções + trechos + pergunta ──► LLM (qwen2.5) ──► resposta com fontes
```

Tudo roda **localmente**: o Ollama serve o modelo de embedding e o modelo de chat; o ChromaDB guarda os vetores em disco.

In [2]:
instalados, faltando = rag.verificar_ollama([config.MODELO_EMBEDDING, config.MODELO_CHAT, config.MODELO_CHAT_PLANO_B])
print("Modelos no Ollama:", sorted(instalados))
print("Faltando:", faltando or "nenhum")
print("Embedding:", config.MODELO_EMBEDDING, "| Chat:", config.MODELO_CHAT, "| Plano B:", config.MODELO_CHAT_PLANO_B)

Modelos no Ollama: ['bge-m3:latest', 'gemma4:26b', 'qwen2.5:1.5b', 'qwen2.5:3b']
Faltando: nenhum
Embedding: bge-m3 | Chat: qwen2.5:3b | Plano B: qwen2.5:1.5b


## Bloco 2 — Indexação

### 2.1 Corpus e metadados escritos à mão
Os PDFs não vêm no repositório: `scripts/01_preparar_corpus.py` baixa cada um do arXiv para `arquivosPDF/artigos/`.

| Arquivo | Artigo |
|---|---|
| `lewis2020_rag.pdf` | Lewis et al. (2020) — https://arxiv.org/abs/2005.11401 |
| `karpukhin2020_dpr.pdf` | Karpukhin et al. (2020) — https://arxiv.org/abs/2004.04906 |
| `gao2023_survey.pdf` | Gao et al. (2023) — https://arxiv.org/abs/2312.10997 |
| `es2023_ragas.pdf` | Es et al. (2023) — https://arxiv.org/abs/2309.15217 |
| `asai2023_selfrag.pdf` | Asai et al. (2023) — https://arxiv.org/abs/2310.11511 |
| `liu2023_lost_middle.pdf` | Liu et al. (2023) — https://arxiv.org/abs/2307.03172 |

Metadado é **decisão de engenharia**: escolhemos campos que viram filtros úteis na busca (`ano`, `tema`, `idioma`).

In [3]:
metadados = rag.carregar_metadados()
pd.DataFrame(metadados)[["arquivo", "titulo", "ano", "veiculo", "tema", "idioma"]]

,arquivo,titulo,ano,veiculo,tema,idioma
0,lewis2020_rag.pdf,Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks,2020,NeurIPS 2020,fundamentos,en
1,karpukhin2020_dpr.pdf,Dense Passage Retrieval for Open-Domain Question Answering,2020,EMNLP 2020,retrieval,en
2,gao2023_survey.pdf,Retrieval-Augmented Generation for Large Language Models: A Survey,2023,arXiv:2312.10997,survey,en
3,es2023_ragas.pdf,Ragas: Automated Evaluation of Retrieval Augmented Generation,2023,EACL 2024 (System Demonstrations),avaliacao,en
4,asai2023_selfrag.pdf,"Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection",2023,ICLR 2024,retrieval,en
5,liu2023_lost_middle.pdf,Lost in the Middle: How Language Models Use Long Contexts,2023,TACL 2024,limitacoes,en


### 2.2 Do PDF ao chunk
Chunking **por página**; páginas longas são subdivididas em pedaços de `TAMANHO_CHUNK` caracteres com `SOBREPOSICAO`
para que uma frase cortada no fim de um chunk apareça inteira no próximo.

In [4]:
paginas = rag.extrair_paginas(config.PASTA_ARTIGOS / "lewis2020_rag.pdf")
print(f"{len(paginas)} páginas; página 3 tem {len(paginas[2])} caracteres")
pedacos = rag.dividir_texto(paginas[2])
for i, pedaco in enumerate(pedacos):
    print(f"chunk {i}: {len(pedaco)} caracteres | início: {pedaco[:70]!r}")
print("\nSobreposição entre chunk 0 e chunk 1:")
print("  fim do 0:   …", pedacos[0][-config.SOBREPOSICAO:])
print("  início do 1:", pedacos[1][:config.SOBREPOSICAO], "…")

19 páginas; página 3 tem 3659 caracteres
chunk 0: 999 caracteres | início: 'byθ that generates a current token based on a context of the previousi'
chunk 1: 998 caracteres | início: 'document as a single latent variable that is marginalized to get the s'
chunk 2: 991 caracteres | início: 'output token, Formally, we define: pRAG-Token(y|x) ≈ N∏ i ∑ z∈top-k(p('
chunk 3: 997 caracteres | início: '[23]. We use a pre-trained bi-encoder from DPR to initialize our retri'
chunk 4: 246 caracteres | início: 'as the parametric memory henceforth. 2.4 Training We jointly train the'

Sobreposição entre chunk 0 e chunk 1:
  fim do 0:   … ieved document as a single latent variable that is marginalized to get the seq2seq probability p(y|x) via a top-K approximation. Concretely, the top K
  início do 1: document as a single latent variable that is marginalized to get the seq2seq probability p(y|x) via a top-K approximation. Concretely, the top K docum …


In [5]:
chunks = rag.gerar_chunks()
contagem = pd.DataFrame([c["metadados"] for c in chunks]).groupby(["arquivo", "tipo_chunk"]).size().unstack(fill_value=0)
print(f"{len(chunks)} chunks no total")
contagem

556 chunks no total


tipo_chunk,pagina,resumo
arquivo,,
asai2023_selfrag.pdf,135,1
es2023_ragas.pdf,39,1
gao2023_survey.pdf,136,1
karpukhin2020_dpr.pdf,71,1
lewis2020_rag.pdf,88,1
liu2023_lost_middle.pdf,81,1


### 2.3 Embeddings + ChromaDB
Cada chunk vira um vetor gerado pelo `bge-m3` (multilíngue) e é gravado com seus metadados.

In [6]:
colecao = rag.abrir_colecao()
if REINDEXAR or colecao.count() != len(chunks):
    inicio = time.perf_counter()
    colecao = rag.indexar(chunks)
    print(f"Indexação em {time.perf_counter() - inicio:.0f}s")
amostra = colecao.get(ids=["es2023_ragas-p001-c00"], include=["metadatas", "embeddings"])
print(f"{colecao.count()} vetores | dimensão {len(amostra['embeddings'][0])}")
{k: (v[:80] + "…" if isinstance(v, str) and len(v) > 80 else v) for k, v in amostra["metadatas"][0].items()}

556 vetores | dimensão 1024


{'tipo_chunk': 'pagina',
 'tema': 'avaliacao',
 'titulo': 'Ragas: Automated Evaluation of Retrieval Augmented Generation',
 'chunk_id': 'es2023_ragas-p001-c00',
 'veiculo': 'EACL 2024 (System Demonstrations)',
 'pagina': 1,
 'autores': 'Shahul Es; Jithin James; Luis Espinosa-Anke; Steven Schockaert',
 'idioma': 'en',
 'resumo': 'Ragas is a framework for evaluating Retrieval Augmented Generation (RAG) systems…',
 'arquivo': 'es2023_ragas.pdf',
 'ano': 2023}

### 2.4 E se o LLM preenchesse os metadados?
Alternativa ao CSV manual: pedir ao LLM um JSON a partir da 1ª página. Comparamos campo a campo com o CSV.

In [7]:
artigo = "es2023_ragas.pdf"
if LLM_AO_VIVO:
    primeira = rag.extrair_paginas(config.PASTA_ARTIGOS / artigo, limpar=False)[0]
    manual = next(m for m in metadados if m["arquivo"] == artigo)
    inicio = time.perf_counter()
    comparacao = rag.comparar_metadados(manual, rag.extrair_metadados_llm(primeira))
    print(f"Extração em {time.perf_counter() - inicio:.1f}s")
else:
    comparacao = carregar_resultado("metadados_llm.json")["comparacao"]
pd.DataFrame(comparacao)

Extração em 158.1s


,campo,csv,llm,igual
0,titulo,Ragas: Automated Evaluation of Retrieval Augmented Generation,Ragas: Automated Evaluation of Retrieval Augmented Generation,True
1,autores,Shahul Es; Jithin James; Luis Espinosa-Anke; Steven Schockaert,Ragas; Shahul Es†; Jithin James†; Luis Espinosa-Anke∗; Steven Schockaert∗,False
2,ano,2023,2023,True
3,veiculo,EACL 2024 (System Demonstrations),N/A (Texto não menciona veículo específico),False
4,tema,avaliacao,retrieval,False
5,idioma,en,en,True


### 2.5 Resumo do abstract gerado pelo LLM
O campo `resumo` do CSV foi gerado assim (no idioma do artigo) e também é indexado como um chunk próprio
(`tipo_chunk = "resumo"`), usado na busca em dois estágios.

In [8]:
primeira = rag.extrair_paginas(config.PASTA_ARTIGOS / "liu2023_lost_middle.pdf", limpar=False)[0]
abstract = rag.extrair_abstract(primeira)
print("ABSTRACT:", abstract[:600], "…\n")
if LLM_AO_VIVO:
    inicio = time.perf_counter()
    resumo = rag.resumir_abstract(abstract, "en")
    print(f"RESUMO AO VIVO ({time.perf_counter() - inicio:.1f}s):", resumo)
print("\nRESUMO NO CSV:", next(m["resumo"] for m in metadados if m["arquivo"] == "liu2023_lost_middle.pdf"))

ABSTRACT: While recent language models have the ability to take long contexts as input, relatively little is known about how well they use longer context. We analyze the performance of language models on two tasks that require identifying relevant information in their input contexts: multi-document question answering and key-value retrieval. We find that performance can degrade significantly when changing the position of relevant information, indicating that current language models do not robustly make use of information in long input contexts. In particular, we observe that performance is often highest …



RESUMO AO VIVO (14.5s): The study finds that recent language models struggle with using longer input contexts effectively, with performance degrading significantly when relevant information is not at the beginning or end of the context. This suggests that current models do not robustly utilize information in long input contexts, even for models explicitly designed to handle such contexts.

RESUMO NO CSV: The study found that recent language models struggle with using longer input contexts effectively, with performance degrading significantly when relevant information is not at the beginning or end of the context. Explicitly long-context models also struggle with accessing relevant information in the middle of long contexts, indicating a need for improved handling of such contexts.


## Bloco 3 — Retrieval top-k na prática
A distância é **cosseno** (0 = idêntico). Observe como a lista cresce com `k` e onde entram trechos menos relevantes.

In [9]:
pergunta = "Quais métricas o Ragas usa para avaliar um pipeline de RAG?"
for k in (1, 4, 8):
    display(Markdown(f"**k = {k}**"), tabela(rag.buscar(pergunta, k=k, colecao=colecao)))

**k = 1**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3727,es2023_ragas.pdf,1,2023,avaliacao,en,"Ragas: Automated Evaluation of Retrieval Augmented Generation Shahul Es†, Jithin James†, Luis Espinosa-Anke∗♢,…"


**k = 4**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3727,es2023_ragas.pdf,1,2023,avaliacao,en,"Ragas: Automated Evaluation of Retrieval Augmented Generation Shahul Es†, Jithin James†, Luis Espinosa-Anke∗♢,…"
1,2,0.4247,es2023_ragas.pdf,1,2023,avaliacao,en,"RAG systems are often evaluated in terms of the language modelling task itself, i.e. by measuring perplexity o…"
2,3,0.4416,gao2023_survey.pdf,14,2023,survey,en,14 TABLE III SUMMARY OF METRICS APPLICABLE FOR EVALUATION ASPECTS OF RAG Context Relevance Faithfulness Answer…
3,4,0.4455,gao2023_survey.pdf,14,2023,survey,en,the evaluation of RAG.These instruments furnish quantitative metrics that not only gauge RAG model performance…


**k = 8**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3727,es2023_ragas.pdf,1,2023,avaliacao,en,"Ragas: Automated Evaluation of Retrieval Augmented Generation Shahul Es†, Jithin James†, Luis Espinosa-Anke∗♢,…"
1,2,0.4247,es2023_ragas.pdf,1,2023,avaliacao,en,"RAG systems are often evaluated in terms of the language modelling task itself, i.e. by measuring perplexity o…"
2,3,0.4416,gao2023_survey.pdf,14,2023,survey,en,14 TABLE III SUMMARY OF METRICS APPLICABLE FOR EVALUATION ASPECTS OF RAG Context Relevance Faithfulness Answer…
3,4,0.4455,gao2023_survey.pdf,14,2023,survey,en,the evaluation of RAG.These instruments furnish quantitative metrics that not only gauge RAG model performance…
4,5,0.4459,gao2023_survey.pdf,12,2023,survey,en,"suitable for RAG. In addition to QA, RAG is continuously being expanded into multiple downstream tasks, such a…"
5,6,0.4551,es2023_ragas.pdf,5,2023,avaliacao,en,"the predictions from the two baselines. For faithfulness, the Ragas prediction are in general highly accurate.…"
6,7,0.4794,es2023_ragas.pdf,5,2023,avaliacao,en,"[context] answer: [answer] Ties, where the same score is assigned by the LLM to both answer candidates, were b…"
7,8,0.4840,gao2023_survey.pdf,2,2023,survey,en,Section V analyzes the three augmentation processes. Section VI focuses on RAG’s downstream tasks and evaluati…


### 3.1 Filtros de metadados (`where`)

In [10]:
pergunta = "Como funciona a recuperação de passagens?"
for nome, filtro in {
    "sem filtro": None,
    "ano >= 2023": {"ano": {"$gte": 2023}},
    "tema = retrieval": {"tema": "retrieval"},
    "idioma = pt": {"idioma": "pt"},
}.items():
    display(Markdown(f"**{nome}** `{filtro}`"), tabela(rag.buscar(pergunta, k=4, where=filtro, colecao=colecao)))

**sem filtro** `None`

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.4500,karpukhin2020_dpr.pdf,12,2020,retrieval,en,"without reindexing the passages during model updates. Our loss function largely follows ORQA’s approach, which…"
1,2,0.4723,karpukhin2020_dpr.pdf,4,2020,retrieval,en,"the answer as the positive passage. If none of the top 100 retrieved passages has the answer, the question wil…"
2,3,0.4752,asai2023_selfrag.pdf,18,2023,retrieval,en,"=Yes, we first retrieve passages using the input and the entire output as queries, to find passages that are r…"
3,4,0.4786,karpukhin2020_dpr.pdf,7,2020,retrieval,en,"Besides the retriever, our QA system consists of a neural reader that outputs the answer to the question. Give…"


**ano >= 2023** `{'ano': {'$gte': 2023}}`

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.4752,asai2023_selfrag.pdf,18,2023,retrieval,en,"=Yes, we first retrieve passages using the input and the entire output as queries, to find passages that are r…"
1,2,0.4962,asai2023_selfrag.pdf,5,2023,retrieval,en,"special token Retrieve =Yes is added, andR retrieves the topK passages, D. For each passage,C further evaluate…"
2,3,0.5027,asai2023_selfrag.pdf,1,2023,retrieval,en,"and preceding generations, SELF -RAG first determines if augmenting the continued generation with retrieved pa…"
3,4,0.5067,gao2023_survey.pdf,11,2023,survey,en,Retrieval aims to enhance the search experience by gradually converging on the most pertinent information thro…


**tema = retrieval** `{'tema': 'retrieval'}`

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.4500,karpukhin2020_dpr.pdf,12,2020,retrieval,en,"without reindexing the passages during model updates. Our loss function largely follows ORQA’s approach, which…"
1,2,0.4723,karpukhin2020_dpr.pdf,4,2020,retrieval,en,"the answer as the positive passage. If none of the top 100 retrieved passages has the answer, the question wil…"
2,3,0.4752,asai2023_selfrag.pdf,18,2023,retrieval,en,"=Yes, we first retrieve passages using the input and the entire output as queries, to find passages that are r…"
3,4,0.4786,karpukhin2020_dpr.pdf,7,2020,retrieval,en,"Besides the retriever, our QA system consists of a neural reader that outputs the answer to the question. Give…"


**idioma = pt** `{'idioma': 'pt'}`

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto


### 3.2 Cross-lingual
Corpus em inglês, pergunta em português: o `bge-m3` coloca as duas línguas no mesmo espaço vetorial.

In [11]:
for texto in ("O desempenho cai quando a informação relevante está no meio de um contexto longo?",
              "Does performance drop when relevant information is in the middle of a long context?"):
    display(Markdown(f"**{texto}**"), tabela(rag.buscar(texto, k=3, colecao=colecao)))

**O desempenho cai quando a informação relevante está no meio de um contexto longo?**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.2939,liu2023_lost_middle.pdf,2,2023,limitacoes,en,or end of the context. We find that changing the position of relevant information in the input context can sub…
1,2,0.2999,liu2023_lost_middle.pdf,1,2023,limitacoes,en,Lost in the Middle: How Language Models Use Long Contexts Nelson F . Liu1∗ Kevin Lin2 John Hewitt1 Ashwin Para…
2,3,0.3317,liu2023_lost_middle.pdf,11,2023,limitacoes,en,"ory. Observing a serial-position-like effect in language models is perhaps surprising, since the selfattention…"


**Does performance drop when relevant information is in the middle of a long context?**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.2697,liu2023_lost_middle.pdf,2,2023,limitacoes,en,or end of the context. We find that changing the position of relevant information in the input context can sub…
1,2,0.2741,liu2023_lost_middle.pdf,1,2023,limitacoes,en,Lost in the Middle: How Language Models Use Long Contexts Nelson F . Liu1∗ Kevin Lin2 John Hewitt1 Ashwin Para…
2,3,0.3169,liu2023_lost_middle.pdf,11,2023,limitacoes,en,"ory. Observing a serial-position-like effect in language models is perhaps surprising, since the selfattention…"


## Bloco 3b — Busca em dois estágios
1. Busca só nos chunks de **resumo** e escolhe os `N_ARTIGOS_ESTAGIO_1` artigos mais próximos.
2. Busca os chunks de página **só dentro desses artigos** (`where={"arquivo": {"$in": [...]}}`).

In [12]:
def comparar_buscas(pergunta):
    display(Markdown(f"### {pergunta}\n**Busca simples**"), tabela(rag.buscar(pergunta, k=4, colecao=colecao)))
    dois = rag.buscar_dois_estagios(pergunta, k=4, colecao=colecao)
    display(Markdown(f"**Estágio 1 — artigos escolhidos** ({dois['caminho']})"),
            tabela(dois["artigos"], colunas=("posicao", "distancia", "arquivo", "titulo")))
    display(Markdown("**Estágio 2 — chunks desses artigos**"), tabela(dois["resultados"]))


comparar_buscas("Como avaliar se a resposta é fiel ao contexto recuperado?")

### Como avaliar se a resposta é fiel ao contexto recuperado?
**Busca simples**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3978,es2023_ragas.pdf,4,2023,avaliacao,en,"inclusion of redundant information. To estimate context relevance, given a question q and its context c(q), th…"
1,2,0.3991,liu2023_lost_middle.pdf,9,2023,limitacoes,en,"the added context and the model’s ability to effectively use long input contexts, but we perform a case study …"
2,3,0.3999,es2023_ragas.pdf,3,2023,avaliacao,en,"estimate answer relevance, for the given answer as(q), we prompt the LLM to generate n potential questions qi …"
3,4,0.4015,es2023_ragas.pdf,3,2023,avaliacao,en,"refers to the idea that the retrieved context should be focused, containing as little irrelevant information a…"


**Estágio 1 — artigos escolhidos** (dois estágios)

,posicao,distancia,arquivo,titulo
0,1,0.4885,es2023_ragas.pdf,Ragas: Automated Evaluation of Retrieval Augmented Generation
1,2,0.4914,asai2023_selfrag.pdf,"Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection"
2,3,0.4915,karpukhin2020_dpr.pdf,Dense Passage Retrieval for Open-Domain Question Answering


**Estágio 2 — chunks desses artigos**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3978,es2023_ragas.pdf,4,2023,avaliacao,en,"inclusion of redundant information. To estimate context relevance, given a question q and its context c(q), th…"
1,2,0.3999,es2023_ragas.pdf,3,2023,avaliacao,en,"estimate answer relevance, for the given answer as(q), we prompt the LLM to generate n potential questions qi …"
2,3,0.4015,es2023_ragas.pdf,3,2023,avaliacao,en,"refers to the idea that the retrieved context should be focused, containing as little irrelevant information a…"
3,4,0.4049,es2023_ragas.pdf,4,2023,avaliacao,en,were fluent in English and were given clear instructions about the meaning of the three considered quality dim…


Quando o estágio 1 **ajuda**: o top-4 fica concentrado no artigo certo (Ragas). Agora um caso em que ele **atrapalha**:
o resumo do *Lost in the Middle* não fala em "mais documentos", então o artigo não passa no estágio 1.

In [13]:
comparar_buscas("Recuperar mais documentos sempre melhora a resposta do modelo?")

### Recuperar mais documentos sempre melhora a resposta do modelo?
**Busca simples**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3033,liu2023_lost_middle.pdf,10,2023,limitacoes,en,5 10 20 30 40 50 Number of Retrieved Docs 50 60 70 80 90Metric claude-1.3 claude-1.3-100k gpt-3.5-turbo-0613 g…
1,2,0.3054,liu2023_lost_middle.pdf,3,2023,limitacoes,en,models with longer input contexts is a trade-off— providing the language model with more information may help …
2,3,0.3273,liu2023_lost_middle.pdf,3,2023,limitacoes,en,retrieved documents—using 50 documents instead of 20 retrieved documents only marginally improves performance …
3,4,0.3413,liu2023_lost_middle.pdf,18,2023,limitacoes,en,total retrieved documents. G.2 20 Total Retrieved Documents Model Index 0 Index 4 Index 9 Index 14 Index 19 Cl…


**Estágio 1 — artigos escolhidos** (dois estágios)

,posicao,distancia,arquivo,titulo
0,1,0.4461,karpukhin2020_dpr.pdf,Dense Passage Retrieval for Open-Domain Question Answering
1,2,0.4588,asai2023_selfrag.pdf,"Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection"
2,3,0.4705,lewis2020_rag.pdf,Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks


**Estágio 2 — chunks desses artigos**

,posicao,distancia,arquivo,pagina,ano,tema,idioma,texto
0,1,0.3475,lewis2020_rag.pdf,8,2020,fundamentos,en,and runtime. Figure 3 (left) shows that retrieving more documents at test time monotonically improves Open-dom…
1,2,0.3742,lewis2020_rag.pdf,8,2020,fundamentos,en,17.9 22.6 56.2 49.4 74.5 90.6RAG-Sequence 44.0 55.8 44.9 53.4 15.3 21.5 57.2 47.5 between these dates and use …
2,3,0.3748,lewis2020_rag.pdf,9,2020,fundamentos,en,language models. Learned Retrieval There is significant work on learning to retrieve documents in information …
3,4,0.3919,lewis2020_rag.pdf,6,2020,fundamentos,en,may perform best because it can generate responses that combine content from several documents. Figure 2 shows…


## Bloco 4 — SHAP: explicando o retrieval

> ⚠️ Aqui explicamos o **retrieval**: quais palavras da pergunta aproximam o vetor da pergunta do vetor de um chunk.
> **Não** é uma explicação do raciocínio do LLM.

A função explicada é `similaridade_cosseno(embedding(pergunta mascarada), embedding(chunk))`. O SHAP mascara palavras
da pergunta e mede quanto a similaridade muda.

In [14]:
import shap

pergunta = "Quais métricas o Ragas usa para avaliar fidelidade e relevância das respostas?"
chunk = rag.buscar(pergunta, k=1, colecao=colecao)[0]
print(f"Chunk explicado: {chunk['arquivo']} p.{chunk['pagina']} (similaridade {chunk['similaridade']:.4f})")
if SHAP_AO_VIVO:
    inicio = time.perf_counter()
    explicacao, funcao = rag.explicar_similaridade(pergunta, chunk["texto"])
    print(f"SHAP em {time.perf_counter() - inicio:.0f}s")
    base, soma, real = explicacao.base_values[0], explicacao.values[0].sum(), funcao([pergunta])[0]
    print(f"base {base:.4f} + soma dos SHAP {soma:+.4f} = {base + soma:.4f} | similaridade real {real:.4f}")
    shap.plots.text(explicacao[0])
else:
    salvo = carregar_resultado("shap_similaridade_chunk1.json")
    print(f"base {salvo['base']:.4f} + soma {sum(salvo['valores']):+.4f} | similaridade real {salvo['similaridade_real']:.4f}")
    display(HTML((config.PASTA_RESULTADOS / "shap_similaridade_chunk1.html").read_text(encoding="utf-8")))

Chunk explicado: es2023_ragas.pdf p.5 (similaridade 0.6456)


SHAP em 48s


base 0.3479 + soma dos SHAP +0.3001 = 0.6480 | similaridade real 0.6456


### 4.1 Shapley dos chunks sobre a resposta (pré-computado)
Para cada subconjunto dos k chunks geramos uma resposta e medimos a similaridade com a resposta que usa todos
(2^k gerações — muito lento para ao vivo). O valor de Shapley de cada chunk é sua contribuição média.
Gerado por `opcional/calcular_shapley_chunks.py`.

In [15]:
shapley = carregar_resultado("shapley_chunks.json")
print("Pergunta:", shapley["pergunta"], "| modelo:", shapley["modelo"])
print(f"v(nenhum chunk) = {shapley['valor_sem_chunks']:.4f} | v(todos) = {shapley['valor_com_todos']:.4f}")
pd.DataFrame(shapley["contribuicoes"]).round(4)

Pergunta: Quais são os tokens de reflexão (reflection tokens) propostos no Self-RAG? | modelo: qwen2.5:3b
v(nenhum chunk) = 0.3270 | v(todos) = 1.0000


,chunk,arquivo,pagina,shapley
0,1,asai2023_selfrag.pdf,10,0.2034
1,2,asai2023_selfrag.pdf,1,0.0113
2,3,gao2023_survey.pdf,12,0.2564
3,4,asai2023_selfrag.pdf,4,0.2019


## Bloco 5 — Prompt augmentation: com e sem contexto
O prompt enriquecido tem duas mensagens: **system** com as instruções (português, só os trechos, citar `[n]`,
responder "não encontrei" quando faltar informação) e **user** com os trechos numerados com fonte + a pergunta.
Com um modelo pequeno (3B), separar as instruções na mensagem de sistema fez diferença: com tudo numa mensagem só,
ele às vezes respondia em inglês ou recusava perguntas que os trechos respondiam.

In [16]:
# TODO(autor): trocar pela pergunta-teste definitiva.
pergunta = "Quais são os tokens de reflexão (reflection tokens) propostos no Self-RAG?"
resultados = rag.buscar(pergunta, k=config.K_PADRAO, colecao=colecao)
print(rag.formatar_mensagens(rag.montar_mensagens(pergunta, resultados))[:1800], "\n[…]")

=== SYSTEM ===
Você é um assistente que responde em português do Brasil usando SOMENTE os trechos fornecidos pelo usuário. Os trechos podem estar em inglês: traduza e explique em português. Se a resposta não estiver nos trechos, responda exatamente: "Não encontrei essa informação nos documentos." Cite a fonte de cada informação com o número do trecho entre colchetes, por exemplo [1].

=== USER ===
Trechos:
[1] (asai2023_selfrag.pdf, p. 10)
higher S&P scores on short-form PopQA, which is consistent with Menick et al. (2022). Human annotators also find ISREL and ISSUP reflection token predictions are mostly aligned with their assessments. Appendix Table 6 shows several annotated examples and explanations on assessments. 6 C ONCLUSION This work introduces SELF -RAG, a new framework to enhance the quality and factuality of LLMs through retrieval on demand and self-reflection. SELF -R AG trains an LM to learn to retrieve, generate, and critique text passages and its own generation by predic

In [17]:
if LLM_AO_VIVO:
    sem = rag.gerar_texto(rag.montar_mensagens(pergunta))
    com = rag.gerar_texto(rag.montar_mensagens(pergunta, resultados))
else:
    salvo = next(r for r in carregar_resultado("com_sem_contexto.json") if r["pergunta"] == pergunta)
    sem, com = salvo["sem_contexto"], salvo["com_contexto"]
# why: mostrar todo o top-k como "Fontes" mistura o que entrou no prompt com o que a resposta de
# fato citou (achado 6.5) — rag.montar_bloco_fontes() usa rag.fontes_da_resposta() por baixo para
# só listar os [n] citados (ou nenhuma, em recusa); os trechos recuperados ficam à parte, já que
# são "o que foi oferecido ao modelo", não "o que ele usou".
fontes_citadas = rag.montar_bloco_fontes(com, resultados).replace("\n\nFontes:\n", "") or "(nenhuma — resposta é uma recusa)"
display(Markdown(f"### Sem contexto\n{sem}\n\n### Com contexto\n{com}\n\n"
                 f"**Fontes citadas**\n```\n{fontes_citadas}\n```\n\n"
                 f"**Trechos enviados ao prompt**\n```\n{rag.formatar_fontes(resultados)}\n```"))

### Sem contexto
Desculpe, mas não tenho informações específicas sobre tokens de reflexão propostos no Self-RAG. Este conceito não parece estar associado a um sistema ou projeto específico conhecido. Você pode estar se referindo a um projeto ou conceito mais amplo que não tenha sido amplamente discutido ou implementado. Se puder fornecer mais contexto ou detalhes, ficarei feliz em tentar ajudar mais.

### Com contexto
O Self-RAG propõe dois tipos de tokens de reflexão: "retrieve" e "critic". O modelo decide quando ativar a busca, ou alternativamente, um limiar pré-definido pode iniciar o processo. Durante a busca, o gerador realiza um corte de fragmentos de busca em múltiplas parágrafos para derivar a sequência mais coerente. Os pontuações de crítico são usadas para atualizar os pontuações de divisão, com a flexibilidade para ajustar esses pesos durante a inferência, ajustando o comportamento do modelo.

**Fontes citadas**
```
[1] asai2023_selfrag.pdf, p. 10
[2] asai2023_selfrag.pdf, p. 1
[3] gao2023_survey.pdf, p. 12
[4] asai2023_selfrag.pdf, p. 4
```

**Trechos enviados ao prompt**
```
[1] asai2023_selfrag.pdf, p. 10
[2] asai2023_selfrag.pdf, p. 1
[3] gao2023_survey.pdf, p. 12
[4] asai2023_selfrag.pdf, p. 4
```

Respostas salvas das outras perguntas-teste (geradas por `scripts/06_com_sem_contexto.py`):

In [18]:
pd.DataFrame(carregar_resultado("com_sem_contexto.json"))[["pergunta", "sem_contexto", "com_contexto"]]

,pergunta,sem_contexto,com_contexto
0,"Segundo o artigo Lost in the Middle, em que posições do contexto os modelos usam melhor a informação?","Segundo o artigo ""Lost in the Middle"", os modelos de linguagem usam melhor a informação nas posições centrais do con...","Segundo o artigo, os modelos de linguagem usam melhor a informação quando ela ocorre na posição inicial ou final do ..."
1,Quais são os tokens de reflexão (reflection tokens) propostos no Self-RAG?,"Desculpe, mas não tenho informações específicas sobre tokens de reflexão propostos no Self-RAG. Este tema não parece...","O Self-RAG propõe dois tipos de tokens de reflexão (reflection tokens): ""retrieve"" e ""critic"". Estes tokens permitem..."
2,Qual é a receita de pão de queijo mineiro?,Aqui está uma receita básica de pão de queijo mineiro:\n\nIngredientes:\n\n- 400g de queijo coalho ralado\n- 2 ovos ...,Não encontrei essa informação nos documentos. [1]


## Bloco 6 — Integração com LLM local usando Ollama
`rag.responder()` busca a resposta em **streaming** e acrescenta as fontes. Para trocar para o plano B, mude
`MODELO_CHAT` em `config.py` (ou passe `modelo=config.MODELO_CHAT_PLANO_B`).

In [19]:
pergunta = "Como o Self-RAG decide quando buscar documentos?"
resultados_bloco6 = rag.buscar(pergunta, colecao=colecao)
salva = config.PASTA_RESULTADOS / "resposta_bloco6.md"
if LLM_AO_VIVO:
    inicio = time.perf_counter()
    saida, texto = display(Markdown("…"), display_id=True), ""
    for pedaco in rag.responder(pergunta, resultados_bloco6):
        texto += pedaco
        saida.update(Markdown(texto))
    print(f"[{time.perf_counter() - inicio:.1f}s com {config.MODELO_CHAT}]")
    salva.write_text(texto, encoding="utf-8")
else:
    display(Markdown(salva.read_text(encoding="utf-8")))
# why: rag.responder() já cita só as fontes usadas (achado 6.5); aqui mostramos também os trechos
# recuperados que foram oferecidos ao modelo, separados e rotulados, para não confundir os dois.
display(Markdown(f"**Trechos enviados ao prompt**\n```\n{rag.formatar_fontes(resultados_bloco6)}\n```"))

O Self-RAG decide quando buscar documentos baseado na probabilidade de geração de termos. Quando essa probabilidade cai abaixo de um certo limiar, o sistema de busca é ativado para coletar informações relevantes, o que melhora o ciclo de busca.

Fontes:
[1] asai2023_selfrag.pdf, p. 1
[2] gao2023_survey.pdf, p. 12
[3] lewis2020_rag.pdf, p. 3
[4] lewis2020_rag.pdf, p. 5

[93.8s com qwen2.5:3b]


**Trechos enviados ao prompt**
```
[1] asai2023_selfrag.pdf, p. 1
[2] gao2023_survey.pdf, p. 12
[3] lewis2020_rag.pdf, p. 3
[4] lewis2020_rag.pdf, p. 5
```

## Bloco 7 — Chatbot com Streamlit
O app reaproveita as mesmas funções de `rag.py`. No terminal, com o `.venv` ativo:

```bash
streamlit run app.py
```

In [20]:
display(Markdown("```python\n" + (RAIZ / "app.py").read_text(encoding="utf-8") + "\n```"))

```python
import streamlit as st

import config
import rag

st.set_page_config(page_title="Assistente RAG — CIIA", page_icon="📚", layout="wide")


@st.cache_resource
def colecao():
    return rag.abrir_colecao()


def montar_filtro(ano_minimo, temas, idiomas):
    condicoes = [{"ano": {"$gte": ano_minimo}}]
    if temas:
        condicoes.append({"tema": {"$in": temas}})
    if idiomas:
        condicoes.append({"idioma": {"$in": idiomas}})
    return rag.combinar_filtros(*condicoes)


def mostrar_fontes(fontes, caminho, artigos):
    with st.expander(f"Fontes ({len(fontes)}) — {caminho}"):
        if artigos:
            st.markdown("**Estágio 1 — artigos escolhidos pelos resumos:** "
                        + ", ".join(f"`{a['arquivo']}`" for a in artigos))
        if not fontes:
            st.info("Nenhum trecho recuperado com esses filtros.")
        for i, fonte in enumerate(fontes, start=1):
            st.markdown(f"**[{i}] {fonte['titulo']}** — `{fonte['arquivo']}`, p. {fonte['pagina']} "
                        f"· distância {fonte['distancia']:.4f} · {fonte['ano']} · {fonte['tema']} · {fonte['idioma']}")
            st.caption(f"Resumo do artigo: {fonte['resumo']}")
            st.text(fonte["texto"][:700])


st.title("📚 Assistente RAG sobre artigos de RAG")
st.caption("Webinário CIIA — Encontro 2 · Ollama + ChromaDB + Streamlit")

with st.sidebar:
    st.header("Configuração")
    modelo = st.selectbox("Modelo de chat", [config.MODELO_CHAT, config.MODELO_CHAT_PLANO_B])
    modo = st.radio("Busca", ["Simples", "Dois estágios"], horizontal=True)
    k = st.slider("k (trechos no contexto)", 1, 10, config.K_PADRAO)
    st.subheader("Filtros de metadados")
    ano_minimo = st.slider("Ano mínimo", 2020, 2026, 2020)
    temas = st.multiselect("Tema", config.TEMAS)
    idiomas = st.multiselect("Idioma", config.IDIOMAS)
    if st.button("Limpar conversa"):
        st.session_state.mensagens = []

try:
    _, faltando = rag.verificar_ollama([config.MODELO_EMBEDDING, modelo])
    if faltando:
        st.warning("Modelos não encontrados no Ollama: " + ", ".join(faltando)
                   + ". Rode `ollama pull <modelo>` no terminal.")
except rag.OllamaIndisponivel as erro:
    st.warning(f"⚠️ {erro}")

if "mensagens" not in st.session_state:
    st.session_state.mensagens = []

for mensagem in st.session_state.mensagens:
    with st.chat_message(mensagem["papel"]):
        st.markdown(mensagem["texto"])
        if mensagem["papel"] == "assistant":
            mostrar_fontes(mensagem["fontes"], mensagem["caminho"], mensagem["artigos"])

pergunta = st.chat_input("Pergunte algo sobre os artigos…")
if pergunta:
    st.session_state.mensagens.append({"papel": "user", "texto": pergunta})
    with st.chat_message("user"):
        st.markdown(pergunta)
    with st.chat_message("assistant"):
        try:
            filtro = montar_filtro(ano_minimo, temas, idiomas)
            if modo == "Dois estágios":
                busca = rag.buscar_dois_estagios(pergunta, k=k, where=filtro, colecao=colecao())
            else:
                busca = {"caminho": "busca simples", "artigos": [],
                         "resultados": rag.buscar(pergunta, k=k, where=filtro, colecao=colecao())}
            st.caption(f"Caminho usado: {busca['caminho']} · k = {k} · modelo {modelo}")
            resposta = st.write_stream(
                rag.responder(pergunta, busca["resultados"], modelo=modelo, incluir_fontes=False)
            )
            mostrar_fontes(busca["resultados"], busca["caminho"], busca["artigos"])
            st.session_state.mensagens.append({
                "papel": "assistant", "texto": resposta, "fontes": busca["resultados"],
                "caminho": busca["caminho"], "artigos": busca["artigos"],
            })
        except rag.OllamaIndisponivel as erro:
            st.error(f"⚠️ {erro}")
            st.session_state.mensagens.pop()

```

## Bloco 8 — Avaliação, reranking e próximos passos
- **RAGAS** (Es et al., 2023): avalia sem resposta de referência — *faithfulness* (a resposta é sustentada pelo
  contexto?), *answer relevancy* (responde à pergunta?) e *context relevance/precision* (os trechos ajudam?).
  `opcional/avaliacao_estilo_ragas.py` implementa essas três ideias com o próprio Ollama como juiz.
- **Reranking**: recuperar top-20 barato e reordenar com um *cross-encoder* antes de mandar top-4 ao LLM.
- **Lost in the Middle** (Liu et al., 2023): mais k nem sempre é melhor — posição do trecho no prompt importa.
- **Escalabilidade**: índices maiores, busca híbrida (BM25 + vetores), cache de embeddings, servidores dedicados.

In [21]:
arquivo = config.PASTA_RESULTADOS / "avaliacao_estilo_ragas.json"
if arquivo.exists():
    display(pd.DataFrame(carregar_resultado(arquivo.name))[["pergunta", "fidelidade", "relevancia_resposta", "precisao_contexto"]].round(2))
else:
    print("Rode opcional/avaliacao_estilo_ragas.py para gerar o resultado.")

,pergunta,fidelidade,relevancia_resposta,precisao_contexto
0,Quais são os tokens de reflexão (reflection tokens) propostos no Self-RAG?,1.0,0.55,0.75
1,"Segundo o artigo Lost in the Middle, em que posições do contexto os modelos usam melhor a informação?",0.6,0.47,0.75
